<a href="https://colab.research.google.com/github/Anshsurana123/dangerous-hybrid-portfolio/blob/main/notebooks/automatic_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hey Snowy — Entraînement Wake Word OpenWakeWord

Notebook one-shot corrigé pour fonctionner sur Google Colab (avril 2026).

**Corrections intégrées par rapport au notebook officiel :**
- `torchaudio.set_audio_backend` deprecated → patché
- `torchaudio.info` supprimé → patché avec soundfile
- `generate_samples.py` manquant → créé avec resampling 16kHz auto
- `piper-sample-generator` installé depuis le bon repo (rhasspy)
- AudioSet (404) → remplacé par FMA dataset
- `onnxscript` manquant → installé explicitement
- Conversion tflite ignorée (non nécessaire pour Android)

⚠️ **Prérequis : activer GPU T4** → Exécution → Modifier le type d'exécution → T4 GPU

⏱️ **Durée totale : ~1h30**

In [13]:
# ============================================================
# CELLULE 0 — Configuration (MODIFIER ICI)
# ============================================================

TARGET_PHRASE = 'jaag_ruut'   # <-- MODIFIE ICI ('hey snowy' ou 'bye bye snowy')
N_SAMPLES = 20000              # Nombre de clips d'entraînement
N_SAMPLES_VAL = 4000          # Nombre de clips de validation
STEPS = 40000                 # Nombre de steps d'entraînement

print(f'Wake word cible : "{TARGET_PHRASE}"')
print(f'Samples : {N_SAMPLES} train / {N_SAMPLES_VAL} val')
print(f'Steps : {STEPS}')

Wake word cible : "jaag_ruut"
Samples : 20000 train / 4000 val
Steps : 40000


In [14]:
# ============================================================
# CELLULE 1 — Installation des dépendances (~5 min)
# ============================================================
import os

# Clone les repos
!git clone https://github.com/dscripka/openWakeWord
!git clone https://github.com/rhasspy/piper-sample-generator

# Télécharge le modèle TTS Piper
!mkdir -p piper-sample-generator/models
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt \
    'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
print('Modèle TTS téléchargé ✅')

# Installe les packages
!pip install -q -e ./openWakeWord
!pip install -q -e ./piper-sample-generator
!pip install -q webrtcvad mutagen torchinfo torchmetrics==1.2.0 speechbrain==0.5.14
!pip install -q audiomentations==0.33.0 torch-audiomentations==0.11.0
!pip install -q acoustics pronouncing datasets==2.14.6 deep-phonemizer==0.0.19
!pip install -q librosa soundfile onnxscript onnx

# Crée le dossier models s'il n'existe pas
os.makedirs('/content/openWakeWord/openwakeword/resources/models', exist_ok=True)

# Télécharge les modèles OpenWakeWord
base_url = 'https://github.com/dscripka/openWakeWord/releases/download/v0.5.1'
models_dir = '/content/openWakeWord/openwakeword/resources/models'
for model_file in ['embedding_model.onnx', 'embedding_model.tflite', 'melspectrogram.onnx', 'melspectrogram.tflite']:
    !wget -q {base_url}/{model_file} -O {models_dir}/{model_file}
    print(f'{model_file} ✅')

print('\nInstallation terminée ✅')

fatal: destination path 'openWakeWord' already exists and is not an empty directory.
Cloning into 'piper-sample-generator'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 184 (delta 70), reused 53 (delta 53), pack-reused 98 (from 1)
Receiving objects: 100% (184/184), 1.04 MiB | 20.91 MiB/s, done.
Resolving deltas: 100% (93/93), done.
Modèle TTS téléchargé ✅
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for openwakeword (pyproject.toml) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 

In [15]:
# ============================================================
# CELLULE 2 — Patches de compatibilité (CRITIQUE)
# ============================================================
import os

# --- Patch 1 : torchaudio.set_audio_backend deprecated ---
!sed -i 's/torchaudio.set_audio_backend("soundfile")/#torchaudio.set_audio_backend("soundfile") # patched/' \
    /usr/local/lib/python3.12/dist-packages/torch_audiomentations/utils/io.py
print('Patch 1 : torchaudio.set_audio_backend ✅')

# --- Patch 2 : torchaudio.info supprimé dans nouvelles versions ---
io_path = '/usr/local/lib/python3.12/dist-packages/torch_audiomentations/utils/io.py'
patch_code = '''# === PATCH torchaudio.info ===
import torchaudio as _torchaudio
import soundfile as _sf
if not hasattr(_torchaudio, 'info'):
    def _torchaudio_info(path):
        class _AudioMetaData:
            def __init__(self, path):
                info = _sf.info(path)
                self.sample_rate = info.samplerate
                self.num_frames = info.frames
                self.num_channels = info.channels
        return _AudioMetaData(path)
    _torchaudio.info = _torchaudio_info
# === FIN PATCH ===
'''

with open(io_path, 'r') as f:
    content = f.read()

if '=== PATCH torchaudio.info ===' not in content:
    with open(io_path, 'w') as f:
        f.write(patch_code + content)
    print('Patch 2 : torchaudio.info ✅')
else:
    print('Patch 2 : déjà appliqué ✅')

# --- Patch 3 : generate_samples.py manquant ---
generate_samples_code = '''import sys, os
import librosa
import soundfile as sf
sys.path.insert(0, "/content/piper-sample-generator")
from piper_sample_generator.__main__ import generate_samples as _generate_samples

def generate_samples(text, max_samples, output_dir, batch_size=1,
                     noise_scales=None, noise_scale_ws=None,
                     length_scales=None, auto_reduce_batch_size=False,
                     file_names=None, **kwargs):
    _generate_samples(
        text=text,
        max_samples=max_samples,
        output_dir=output_dir,
        model="/content/piper-sample-generator/models/en_US-libritts_r-medium.pt",
        batch_size=batch_size,
        noise_scales=noise_scales or [0.667],
        noise_scale_ws=noise_scale_ws or [0.8],
        length_scales=length_scales or [0.75, 1.0, 1.25],
        file_names=file_names,
    )
    # Resample tous les fichiers générés à 16kHz
    for fname in os.listdir(output_dir):
        if fname.endswith(".wav"):
            path = os.path.join(output_dir, fname)
            audio, sr = librosa.load(path, sr=16000)
            sf.write(path, audio, 16000)
'''

with open('/content/openWakeWord/openwakeword/generate_samples.py', 'w') as f:
    f.write(generate_samples_code)
print('Patch 3 : generate_samples.py ✅')
# --- Patch 4 : PyTorch 2.6 weights_only ---
!sed -i 's/torch.load(checkpoint_path, map_location=device)/torch.load(checkpoint_path, map_location=device, weights_only=False)/g' \
    /content/openWakeWord/openwakeword/data.py
!sed -i 's/torch.load(checkpoint_path, map_location=device)/torch.load(checkpoint_path, map_location=device, weights_only=False)/g' \
    /usr/local/lib/python3.12/dist-packages/dp/model/model.py
print('Patch 4 : weights_only=False ✅')

# --- Patch 5 : dynamo=False (prevents .onnx.data file) ---
with open('/content/openWakeWord/openwakeword/train.py', 'r') as f:
    _t = f.read()
if 'dynamo=False' not in _t:
    _t = _t.replace(
        'os.path.join(output_dir, model_name + ".onnx"), opset_version=13)',
        'os.path.join(output_dir, model_name + ".onnx"), opset_version=13, dynamo=False)'
    )
    with open('/content/openWakeWord/openwakeword/train.py', 'w') as f:
        f.write(_t)
print('Patch 5 : dynamo=False ✅')

# --- Patch 6 : use train features for validation (test features have wrong shape) ---
with open('/content/openWakeWord/openwakeword/train.py', 'r') as f:
    _t = f.read()
_t = _t.replace(
    'np.load(os.path.join(feature_save_dir, "positive_features_test.npy")).shape[1:]',
    'np.load(os.path.join(feature_save_dir, "positive_features_train.npy")).shape[1:]'
)
_t = _t.replace(
    'X_val_neg = np.load(os.path.join(feature_save_dir, "negative_features_test.npy"))',
    'X_val_neg = np.load(os.path.join(feature_save_dir, "negative_features_train.npy"))'
)
_t = _t.replace(
    'X_val_pos = np.load(os.path.join(feature_save_dir, "positive_features_test.npy"))',
    'X_val_pos = np.load(os.path.join(feature_save_dir, "positive_features_train.npy"))'
)
with open('/content/openWakeWord/openwakeword/train.py', 'w') as f:
    f.write(_t)
print('Patch 6 : validation shape fix ✅')

print('\nTous les patches appliqués ✅')
print('\nTous les patches appliqués ✅')

Patch 1 : torchaudio.set_audio_backend ✅
Patch 2 : torchaudio.info ✅
Patch 3 : generate_samples.py ✅
Patch 4 : weights_only=False ✅
Patch 5 : dynamo=False ✅
Patch 6 : validation shape fix ✅

Tous les patches appliqués ✅

Tous les patches appliqués ✅


In [16]:
# ============================================================
# CELLULE 3 — Téléchargement données de fond FMA (~1 min)
# ============================================================
import os, numpy as np, scipy.io.wavfile, datasets
from tqdm import tqdm

output_dir = './fma'
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

if len(os.listdir(output_dir)) < 100:
    print('Téléchargement FMA dataset...')
    fma_dataset = datasets.load_dataset('rudraml/fma', name='small', split='train', streaming=True)
    fma_dataset = iter(fma_dataset.cast_column('audio', datasets.Audio(sampling_rate=16000)))
    n_hours = 1
    for i in tqdm(range(n_hours*3600//30)):
        row = next(fma_dataset)
        name = row['audio']['path'].split('/')[-1].replace('.mp3', '.wav')
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000,
                               (row['audio']['array']*32767).astype(np.int16))
    print(f'FMA téléchargé : {len(os.listdir(output_dir))} fichiers ✅')
else:
    print(f'FMA déjà présent : {len(os.listdir(output_dir))} fichiers ✅')

Téléchargement FMA dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


100%|██████████| 120/120 [00:51<00:00,  2.35it/s]

FMA téléchargé : 120 fichiers ✅


In [17]:
# ============================================================
# CELLULE 4 — Téléchargement MIT RIRs (~1 min)
# ============================================================
import os, numpy as np, scipy.io.wavfile, datasets
from tqdm import tqdm

os.makedirs('./mit_rirs', exist_ok=True)

if len(os.listdir('./mit_rirs')) < 50:
    print('Téléchargement MIT RIRs...')
    rir_dataset = datasets.load_dataset(
        'davidscripka/MIT_environmental_impulse_responses',
        split='train', streaming=True
    )
    rir_dataset = iter(rir_dataset.cast_column('audio', datasets.Audio(sampling_rate=16000)))
    for i in tqdm(range(100)):
        try:
            row = next(rir_dataset)
            scipy.io.wavfile.write(
                f'./mit_rirs/rir_{i:04d}.wav', 16000,
                (row['audio']['array']*32767).astype(np.int16)
            )
        except StopIteration:
            break
    print(f'MIT RIRs : {len(os.listdir("./mit_rirs"))} fichiers ✅')
else:
    print(f'MIT RIRs déjà présents : {len(os.listdir("./mit_rirs"))} fichiers ✅')

Téléchargement MIT RIRs...


Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

100%|██████████| 100/100 [00:13<00:00,  7.54it/s]

MIT RIRs : 100 fichiers ✅


In [18]:
# ============================================================
# CELLULE 5 — Téléchargement features ACAV100M (~2 min, 16GB)
# ============================================================
import os

if not os.path.exists('openwakeword_features_ACAV100M_2000_hrs_16bit.npy'):
    print('Téléchargement features ACAV100M (16GB)...')
    !wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
    print('Features ACAV100M ✅')
else:
    print('Features ACAV100M déjà présentes ✅')

if not os.path.exists('validation_set_features.npy'):
    print('Téléchargement features validation...')
    !wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
    print('Features validation ✅')
else:
    print('Features validation déjà présentes ✅')

Téléchargement features ACAV100M (16GB)...
Features ACAV100M ✅
Téléchargement features validation...
Features validation ✅


In [19]:
# ============================================================
# CELLULE 6 — Configuration du modèle
# ============================================================
import yaml, sys
sys.path.insert(0, '/content/openWakeWord')
sys.path.insert(0, '/content/piper-sample-generator')

MODEL_NAME = TARGET_PHRASE.replace(' ', '_')

config = yaml.load(open('openWakeWord/examples/custom_model.yml', 'r').read(), yaml.Loader)
config['target_phrase'] = [TARGET_PHRASE]
config['model_name'] = MODEL_NAME
config['n_samples'] = N_SAMPLES
config['n_samples_val'] = N_SAMPLES_VAL
config['steps'] = STEPS
config['target_accuracy'] = 0.6
config['target_recall'] = 0.25
config['background_paths'] = ['./fma']
config['rir_paths'] = ['./mit_rirs']
config['feature_data_files'] = {'ACAV100M_sample': 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy'}
config['false_positive_validation_data_path'] = 'validation_set_features.npy'
config['piper_model'] = '/content/piper-sample-generator/models/en_US-libritts_r-medium.pt'

with open('my_model.yaml', 'w') as f:
    yaml.dump(config, f)

print(f'Config sauvegardée pour : "{TARGET_PHRASE}" ✅')

Config sauvegardée pour : "jaag_ruut" ✅


In [20]:
# ============================================================
# CELLULE 7 — Génération des clips (~15 min)
# ============================================================
import subprocess, sys

print('Génération des clips positifs et négatifs...')
result = subprocess.run(
    [sys.executable, 'openWakeWord/openwakeword/train.py',
     '--training_config', 'my_model.yaml', '--generate_clips'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
# Affiche juste les lignes INFO
for line in result.stdout.split('\n'):
    if 'INFO' in line or 'Done' in line or 'ERROR' in line or 'Traceback' in line:
        print(line)

if result.returncode == 0:
    print('\n✅ Génération terminée !')
else:
    print(f'\n❌ Erreur code {result.returncode}')
    print(result.stdout[-2000:])

Génération des clips positifs et négatifs...
INFO:root:##################################################
INFO:piper_sample_generator.__main__:Successfully loaded the model
INFO:piper_sample_generator.__main__:Done
INFO:root:##################################################
INFO:piper_sample_generator.__main__:Successfully loaded the model
INFO:piper_sample_generator.__main__:Done
INFO:root:##################################################
INFO:piper_sample_generator.__main__:Successfully loaded the model
INFO:piper_sample_generator.__main__:Done
INFO:root:##################################################
INFO:piper_sample_generator.__main__:Successfully loaded the model
INFO:piper_sample_generator.__main__:Done

✅ Génération terminée !


In [21]:
# ============================================================
# CELLULE 8 — Augmentation + calcul features (~5 min)
# ============================================================
import subprocess, sys

print('Augmentation et calcul des features...')
result = subprocess.run(
    [sys.executable, 'openWakeWord/openwakeword/train.py',
     '--training_config', 'my_model.yaml', '--augment_clips'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
for line in result.stdout.split('\n'):
    if 'INFO' in line or 'WARNING' in line or 'ERROR' in line or 'Traceback' in line or '%' in line:
        print(line)

if result.returncode == 0:
    import os
    npy_files = [f for f in os.listdir(f'./my_custom_model/{MODEL_NAME}') if f.endswith('.npy')]
    print(f'\n✅ Augmentation terminée ! Features: {npy_files}')
else:
    print(f'\n❌ Erreur code {result.returncode}')
    print(result.stdout[-2000:])

Augmentation et calcul des features...

✅ Augmentation terminée ! Features: ['positive_features_test.npy', 'negative_features_test.npy', 'negative_features_train.npy', 'positive_features_train.npy']


In [25]:
with open('/content/openWakeWord/openwakeword/train.py', 'r') as f:
    _t = f.read()

# Fix vstack shape mismatch by truncating to 16 steps
_t = _t.replace(
    'torch.from_numpy(np.vstack((X_val_pos, X_val_neg))),',
    'torch.from_numpy(np.vstack((X_val_pos[:, :16, :], X_val_neg[:, :16, :]))),\n'
)

with open('/content/openWakeWord/openwakeword/train.py', 'w') as f:
    f.write(_t)
print('Patched! Rerun Cell 9.')

Patched! Rerun Cell 9.


In [27]:
with open('/content/openWakeWord/openwakeword/data.py', 'r') as f:
    _d = f.read()

_d = _d.replace(
    'return np.vstack(X), np.array(y)',
    'X_fixed = [x[:16, :] if x.shape[0] > 16 else x for x in X]\n        return np.vstack(X_fixed), np.array(y)'
)

with open('/content/openWakeWord/openwakeword/data.py', 'w') as f:
    f.write(_d)
print('Patched data.py ✅')

Patched data.py ✅


In [ ]:
# ============================================================
# CELLULE 9 — Entraînement du modèle (~20-30 min)
# ============================================================
import subprocess, sys

print('Entraînement en cours...')
result = subprocess.run(
    [sys.executable, 'openWakeWord/openwakeword/train.py',
     '--training_config', 'my_model.yaml', '--train_model'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
# Affiche résultats finaux
lines = result.stdout.split('\n')
for line in lines:
    if any(x in line for x in ['Final Model', 'Saving ONNX', 'INFO:root:####', 'Accuracy', 'Recall', 'False Pos']):
        print(line)

import os
onnx_path = f'./my_custom_model/{MODEL_NAME}.onnx'
if os.path.exists(onnx_path):
    size = os.path.getsize(onnx_path)
    print(f'\n✅ Modèle ONNX généré : {onnx_path} ({size/1024:.1f} KB)')
elif result.returncode != 0:
    # L'erreur tflite est normale et ignorée
    if 'onnx_tf' in result.stdout:
        print('\n⚠️ Erreur tflite ignorée (non nécessaire pour Android)')
        print('Vérification du fichier ONNX...')
        all_files = os.listdir('./my_custom_model')
        print('Fichiers:', all_files)
    else:
        print(f'\n❌ Erreur code {result.returncode}')
        print(result.stdout[-3000:])

Entraînement en cours...


In [1]:
import os, shutil
from google.colab import drive, files

drive.mount('/content/drive')

onnx_path = f'./my_custom_model/{MODEL_NAME}.onnx'
data_path = f'./my_custom_model/{MODEL_NAME}.onnx.data'
drive_out = f'/content/drive/MyDrive/jaag_ruut_final.onnx'

if not os.path.exists(onnx_path):
    print('❌ ONNX not found')
else:
    if os.path.exists(data_path):
        print('.data file found — merging...')
        import onnx
        model = onnx.load(onnx_path, load_external_data=True)
        onnx.save_model(model, '/content/my_custom_model/merged.onnx', save_as_external_data=False)
        shutil.copy('/content/my_custom_model/merged.onnx', drive_out)
        files.download('/content/my_custom_model/merged.onnx')
    else:
        shutil.copy(onnx_path, drive_out)
        files.download(onnx_path)
    print(f'✅ Saved to Drive and downloaded!')

Mounted at /content/drive


NameError: name 'MODEL_NAME' is not defined

In [24]:
# ============================================================
# CELLULE 11 — Téléchargement modèles de base OpenWakeWord
# (nécessaires pour l'intégration Android)
# ============================================================
from google.colab import files
import os

base_models = [
    '/content/openWakeWord/openwakeword/resources/models/melspectrogram.onnx',
    '/content/openWakeWord/openwakeword/resources/models/embedding_model.onnx',
]

for path in base_models:
    if os.path.exists(path):
        files.download(path)
        print(f'✅ {os.path.basename(path)} téléchargé')
    else:
        print(f'❌ Non trouvé : {path}')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ melspectrogram.onnx téléchargé


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ embedding_model.onnx téléchargé
